### Possible template optimization model as the starting point for our project.

You work at a media company, and you want to sell advertising spots for the upcoming NFL season. You project ratings for the prospective customers, and price the package based on the ratings.

**4 ratings projection scenarios:**

* conservative: 1,800,000 viewers, price $90,000
* medium: 2,100,000 viewers, price $100,000
* premium: 2,500,000 viewers, price $110,000
* aggressive 3,000,000 viewers, price $125,000

**Each ratings projection assumes a CPM, or cost per thosuand views:**

* conservative: $50.00
* medium: $47.62
* premium: $44.00
* aggressive: $41.67

**Each ratings projection has a probability associated with achieving at least that many views:**

* conservative: 80%
* medium: 60%
* premium: 45%
* aggressive: 35%

Next, you have your prospects list, including their name, their goal CPM, and their underdelivery penalty. If you sell a certain package to a customer, and don't deliver on the projected ratings, you owe them the difference between the viewer projection and the actual viewers, times the underdelivery penalty amount.

You have 20 ad spots to sell. You need maximize revenue by matching the highest projections possible for the clients who are willing to pay at least that much CPM, while minimizing the expected underdelivery penalty based on the probability that the scenario will occur.

| ID  | Prospect               | Goal CPM | Underdelivery Penalty | Underdelivery Factor |
| --- | ---------------------- | -------: | --------------------: | -------------------: |
| C01 | Car Dealer A           |    42.71 |                  0.05 |                 0.95 |
| C02 | Car Dealer B           |    43.33 |                  0.15 |                 0.85 |
| C03 | Grocery Store A        |    40.24 |                   0.1 |                  0.9 |
| C04 | Specialty Retail A     |    43.95 |                  0.05 |                 0.95 |
| C05 | Fast Food A            |    44.57 |                  0.05 |                 0.95 |
| C06 | Fast Food B            |    45.19 |                  0.25 |                 0.75 |
| C07 | Grocery Store B        |     45.8 |                  0.05 |                 0.95 |
| C08 | Car Dealer C           |    46.43 |                  0.15 |                 0.85 |
| C09 | Fast Food C            |    40.85 |                   0.1 |                  0.9 |
| C10 | Sporting Goods A       |    47.04 |                  0.05 |                 0.95 |
| C11 | Farm Equipment A       |    47.67 |                   0.1 |                  0.9 |
| C12 | Home Improvement A     |    48.28 |                  0.25 |                 0.75 |
| C13 | Bank A                 |     48.9 |                  0.05 |                 0.95 |
| C14 | Furniture Store A      |    49.52 |                   0.2 |                  0.8 |
| C15 | Auto Parts A           |    41.48 |                  0.25 |                 0.75 |
| C16 | Insurance Agency A     |    50.14 |                   0.2 |                  0.8 |
| C17 | HVAC Company A         |    50.76 |                  0.05 |                 0.95 |
| C18 | Restaurant A           |    51.38 |                   0.1 |                  0.9 |
| C19 | Farm Supply A          |    51.99 |                   0.1 |                  0.9 |
| C20 | Credit Union A         |    52.62 |                  0.05 |                 0.95 |
| C21 | Outdoor Recreation A   |    53.23 |                   0.2 |                  0.8 |
| C22 | Beverage Distributor A |    42.09 |                   0.1 |                  0.9 |
| C23 | Car Dealer D           |    53.86 |                  0.25 |                 0.75 |
| C24 | Grocery Store A        |    54.47 |                   0.2 |                  0.8 |
| C25 | Home Builder A         |    55.09 |                  0.15 |                 0.85 |
| C26 | Healthcare System A    |    55.71 |                  0.25 |                 0.75 |
| C27 | Wireless Retailer A    |    56.33 |                  0.25 |                 0.75 |
| C28 | Convenience Store A    |     42.4 |                  0.25 |                 0.75 |
| C29 | Ag Services A          |    57.57 |                  0.15 |                 0.85 |
| C30 | Financial Services A   |    58.81 |                   0.2 |                  0.8 |


In [ ]:
import math
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import course_utils  # course helpers (print_status, print_model, ...)

In [ ]:
df_p = pd.read_csv("prospects_list.csv")

df_p

In [4]:
import gurobipy as gp
from gurobipy import GRB


# ============================================================
# 1. INDEX SETS
# ============================================================

customers = [
    "C01", "C02", "C03", "C04", "C05",
    "C06", "C07", "C08", "C09", "C10",
    "C11", "C12", "C13", "C14", "C15",
    "C16", "C17", "C18", "C19", "C20",
    "C21", "C22", "C23", "C24", "C25",
    "C26", "C27", "C28", "C29", "C30"
]

scenarios = [
    "conservative",
    "medium",
    "premium",
    "aggressive"
]


# ============================================================
# 2. CUSTOMER PARAMETERS
# ============================================================

goal_cpm = {
    "C01": 42.71,
    "C02": 43.33,
    "C03": 40.24,
    "C04": 43.95,
    "C05": 44.57,
    "C06": 45.19,
    "C07": 45.80,
    "C08": 46.43,
    "C09": 40.85,
    "C10": 47.04,
    "C11": 47.67,
    "C12": 48.28,
    "C13": 48.90,
    "C14": 49.52,
    "C15": 41.48,
    "C16": 50.14,
    "C17": 50.76,
    "C18": 51.38,
    "C19": 51.99,
    "C20": 52.62,
    "C21": 53.23,
    "C22": 42.09,
    "C23": 53.86,
    "C24": 54.47,
    "C25": 55.09,
    "C26": 55.71,
    "C27": 56.33,
    "C28": 42.40,
    "C29": 57.57,
    "C30": 58.81
}

underdelivery_penalty = {
    "C01": 0.05,
    "C02": 0.15,
    "C03": 0.10,
    "C04": 0.05,
    "C05": 0.05,
    "C06": 0.25,
    "C07": 0.05,
    "C08": 0.15,
    "C09": 0.10,
    "C10": 0.05,
    "C11": 0.10,
    "C12": 0.25,
    "C13": 0.05,
    "C14": 0.20,
    "C15": 0.25,
    "C16": 0.20,
    "C17": 0.05,
    "C18": 0.10,
    "C19": 0.10,
    "C20": 0.05,
    "C21": 0.20,
    "C22": 0.10,
    "C23": 0.25,
    "C24": 0.20,
    "C25": 0.15,
    "C26": 0.25,
    "C27": 0.25,
    "C28": 0.25,
    "C29": 0.15,
    "C30": 0.20
}


# ============================================================
# 3. AD PACKAGE PARAMETERS
# ============================================================

views = {
    "conservative": 1_800_000,
    "medium": 2_100_000,
    "premium": 2_500_000,
    "aggressive": 3_000_000
}

price = {
    "conservative": 90_000,
    "medium": 100_000,
    "premium": 110_000,
    "aggressive": 125_000
}

cpm = {
    "conservative": 50.00,
    "medium": 47.62,
    "premium": 44.00,
    "aggressive": 41.67
}


# ============================================================
# 4. ACTUAL VIEWER OUTCOME DISTRIBUTION
# ============================================================

# The original probabilities were cumulative:
#
# P(Viewers >= 1.8M) = 80%
# P(Viewers >= 2.1M) = 60%
# P(Viewers >= 2.5M) = 45%
# P(Viewers >= 3.0M) = 35%
#
# Therefore:
#
# Below 1.8M      = 20%
# 1.8M - 2.1M    = 20%
# 2.1M - 2.5M    = 15%
# 2.5M - 3.0M    = 10%
# 3.0M+          = 35%
#
# We use 1.5M as the representative outcome
# for the "below 1.8M" category.


actual_viewers = {
    1_500_000: 0.20,
    1_800_000: 0.20,
    2_100_000: 0.15,
    2_500_000: 0.10,
    3_000_000: 0.35
}


# Verify that probabilities equal 100%

assert abs(sum(actual_viewers.values()) - 1.0) < 0.000001


# ============================================================
# 5. DETERMINE PACKAGE ELIGIBILITY
# ============================================================

# eligible[c,s] = 1 if customer c's goal CPM is high enough
# to accept package s.
#
# Otherwise eligible[c,s] = 0.

eligible = {}

for c in customers:
    for s in scenarios:

        if cpm[s] <= goal_cpm[c]:
            eligible[c, s] = 1
        else:
            eligible[c, s] = 0


# ============================================================
# 6. CALCULATE EXPECTED UNDERDELIVERY PENALTY
# ============================================================

expected_penalty = {}

for c in customers:
    for s in scenarios:

        expected_penalty[c, s] = sum(
            probability *
            max(views[s] - actual, 0) *
            underdelivery_penalty[c]
            for actual, probability
            in actual_viewers.items()
        )


# ============================================================
# 7. CALCULATE EXPECTED PROFIT
# ============================================================

expected_profit = {}

for c in customers:
    for s in scenarios:

        expected_profit[c, s] = (
            price[s]
            - expected_penalty[c, s]
        )


# ============================================================
# 8. CREATE GUROBI MODEL
# ============================================================

model = gp.Model("NFL_Advertising_Optimization")


# ============================================================
# 9. DECISION VARIABLES
# ============================================================

# x[c,s] = 1 if customer c receives package s
#          0 otherwise

x = model.addVars(
    customers,
    scenarios,
    vtype=GRB.BINARY,
    name="x"
)


# ============================================================
# 10. CONSTRAINTS
# ============================================================

# ------------------------------------------------------------
# Constraint 1:
# Each customer can receive at most one package.
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(x[c, s] for s in scenarios) <= 1
        for c in customers
    ),
    name="one_package_per_customer"
)


# ------------------------------------------------------------
# Constraint 2:
# Package CPM cannot exceed customer's goal CPM.
# ------------------------------------------------------------

model.addConstrs(
    (
        x[c, s] <= eligible[c, s]
        for c in customers
        for s in scenarios
    ),
    name="cpm_requirement"
)


# ------------------------------------------------------------
# Constraint 3:
# We have exactly 20 advertising spots available.
# ------------------------------------------------------------

model.addConstr(
    gp.quicksum(
        x[c, s]
        for c in customers
        for s in scenarios
    ) <= 20,
    name="advertising_spots"
)


# ============================================================
# 11. OBJECTIVE FUNCTION
# ============================================================

# Maximize expected profit:
#
# Revenue from the package
# minus
# Expected underdelivery penalty

model.setObjective(
    gp.quicksum(
        expected_profit[c, s] * x[c, s]
        for c in customers
        for s in scenarios
    ),
    GRB.MAXIMIZE
)


# ============================================================
# 12. SOLVE MODEL
# ============================================================

model.optimize()


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

if model.status == GRB.OPTIMAL:

    print("\nOPTIMAL ADVERTISING PLAN")
    print("=" * 75)

    total_revenue = 0
    total_expected_penalty = 0

    for c in customers:
        for s in scenarios:

            if x[c, s].X > 0.5:

                revenue = price[s]
                penalty = expected_penalty[c, s]
                profit = expected_profit[c, s]

                total_revenue += revenue
                total_expected_penalty += penalty

                print(
                    f"{c}: "
                    f"{s.capitalize():12} | "
                    f"Views: {views[s]:>9,} | "
                    f"CPM: ${cpm[s]:>5.2f} | "
                    f"Revenue: ${revenue:>9,.0f} | "
                    f"Expected Penalty: ${penalty:>9,.2f} | "
                    f"Expected Profit: ${profit:>9,.2f}"
                )

    print("=" * 75)

    print(f"Total Revenue:       ${total_revenue:,.2f}")
    print(f"Expected Penalties:  ${total_expected_penalty:,.2f}")
    print(f"Expected Profit:     ${model.ObjVal:,.2f}")

else:

    print("No optimal solution found.")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-8250U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2685060 - for non-commercial use only - registered to sa___@uiowa.edu
Optimize a model with 151 rows, 120 columns and 360 nonzeros
Model fingerprint: 0x0cc36077
Variable types: 0 continuous, 120 integer (120 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+04, 9e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 1441250.0000
Presolve removed 151 rows and 120 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 1.6695e+06 1.44125e+06 

Optimal solution found (tolerance 1.00e-04)
B